# Data Quality com Great Expectations (Google Colab)

Este notebook executa GX Core no runtime do Colab, compatível com Python 3.10–3.13. Ele valida os três CSVs principais sem alterar ou reenviar os arquivos à Dadosfera. Opcionalmente, valida `tb_reviews_aligned_synthetic.csv`.

No Colab, envie `tb_orders.csv`, `tb_order_items.csv` e `tb_products_unstructured.csv`; se disponível, envie também `tb_reviews_aligned_synthetic.csv`. No VS Code, o notebook lê esses arquivos diretamente de `data/bronze/` e `data/silver/`. Ao final, gera `data_quality_report.json` e `log_execucao.csv`; no Colab, ambos são baixados automaticamente.

In [3]:
%pip install -q "great_expectations>=1.0,<2.0" pandas

Note: you may need to restart the kernel to use updated packages.


In [5]:
from datetime import datetime, timezone
from io import BytesIO
import json
import os
from pathlib import Path

import great_expectations as gx
import pandas as pd

# 1. Registro do horário de início
started_at = datetime.now(timezone.utc)

# O marcador do runtime evita confundir instalações locais do pacote com o Colab.
IN_COLAB = "COLAB_RELEASE_TAG" in os.environ

if IN_COLAB:
    from google.colab import files

    print("Executando no Google Colab. Selecione os CSVs para validação...")
    uploaded = files.upload()

    def load_uploaded_csv(name):
        return pd.read_csv(BytesIO(uploaded[name])) if name in uploaded else None

    orders = load_uploaded_csv("tb_orders.csv")
    items = load_uploaded_csv("tb_order_items.csv")
    products = load_uploaded_csv("tb_products_unstructured.csv")
    reviews = load_uploaded_csv("tb_reviews_aligned_synthetic.csv")

    if orders is None or items is None or products is None:
        raise FileNotFoundError("Envie tb_orders.csv, tb_order_items.csv e tb_products_unstructured.csv.")

    BASE_DIR = Path(".")
else:
    # Identificação da raiz do projeto local (VS Code)
    if (Path("..") / "data" / "bronze").exists():
        BASE_DIR = Path("..")
    elif (Path(".") / "data" / "bronze").exists():
        BASE_DIR = Path(".")
    else:
        BASE_DIR = Path(".")

    BRONZE_DIR = BASE_DIR / "data" / "bronze"
    SILVER_DIR = BASE_DIR / "data" / "silver"

    orders_path = BRONZE_DIR / "tb_orders.csv"
    items_path = BRONZE_DIR / "tb_order_items.csv"
    products_path = BRONZE_DIR / "tb_products_unstructured.csv"
    reviews_path = SILVER_DIR / "tb_reviews_aligned_synthetic.csv"

    required_paths = [orders_path, items_path, products_path]
    missing_paths = [path for path in required_paths if not path.exists()]

    if missing_paths:
        raise FileNotFoundError(
            f"Arquivos principais ausentes nos caminhos especificados: {[str(path) for path in missing_paths]}"
        )

    orders = pd.read_csv(orders_path)
    items = pd.read_csv(items_path)
    products = pd.read_csv(products_path)
    reviews = pd.read_csv(reviews_path) if reviews_path.exists() else None

# Parsing estrito: valores ausentes ou malformados viram NaN/NaT e falham nos checks abaixo.
items["price_parsed"] = pd.to_numeric(items["price"], errors="coerce")
items["freight_value_parsed"] = pd.to_numeric(items["freight_value"], errors="coerce")
orders["purchase_timestamp_parsed"] = pd.to_datetime(
    orders["order_purchase_timestamp"], errors="coerce"
)

# Strings vazias ou compostas apenas por espaços também devem falhar.
products["title_clean"] = products["title_raw"].fillna("").astype(str).str.strip()
products["description_clean"] = products["description_raw"].fillna("").astype(str).str.strip()

# 2. Inicialização do Great Expectations (Modo Efêmero)
context = gx.get_context(mode="ephemeral")
checks = []

def make_batch(table_name, dataframe):
    data_source = context.data_sources.add_pandas(name=f"{table_name}_source")
    data_asset = data_source.add_dataframe_asset(name=f"{table_name}_asset")
    batch_definition = data_asset.add_batch_definition_whole_dataframe(name=f"{table_name}_batch")
    return batch_definition.get_batch(batch_parameters={"dataframe": dataframe})

def run_expectations(table_name, dataframe, expectations):
    batch = make_batch(table_name, dataframe)
    for expectation_name, expectation in expectations:
        result = batch.validate(expectation)
        checks.append({
            "table": table_name,
            "expectation": expectation_name,
            "success": bool(result.success)
        })

# 3. Execução das Validações
run_expectations("orders", orders, [
    ("order_id_not_null", gx.expectations.ExpectColumnValuesToNotBeNull(column="order_id")),
    ("order_id_unique", gx.expectations.ExpectColumnValuesToBeUnique(column="order_id")),
    ("customer_id_not_null", gx.expectations.ExpectColumnValuesToNotBeNull(column="customer_id")),
    ("purchase_timestamp_not_null", gx.expectations.ExpectColumnValuesToNotBeNull(column="order_purchase_timestamp")),
    ("purchase_timestamp_valid_datetime", gx.expectations.ExpectColumnValuesToNotBeNull(column="purchase_timestamp_parsed")),
    ("order_status_domain", gx.expectations.ExpectColumnValuesToBeInSet(
        column="order_status", value_set=["delivered", "shipped", "processing", "canceled"]
    )),
])

run_expectations("items", items, [
    ("order_id_not_null", gx.expectations.ExpectColumnValuesToNotBeNull(column="order_id")),
    ("product_id_not_null", gx.expectations.ExpectColumnValuesToNotBeNull(column="product_id")),
    ("price_valid_numeric", gx.expectations.ExpectColumnValuesToNotBeNull(column="price_parsed")),
    ("price_nonnegative", gx.expectations.ExpectColumnValuesToBeBetween(column="price_parsed", min_value=0)),
    ("freight_valid_numeric", gx.expectations.ExpectColumnValuesToNotBeNull(column="freight_value_parsed")),
    ("freight_nonnegative", gx.expectations.ExpectColumnValuesToBeBetween(column="freight_value_parsed", min_value=0)),
])

run_expectations("products", products, [
    ("product_id_not_null", gx.expectations.ExpectColumnValuesToNotBeNull(column="product_id")),
    ("product_id_unique", gx.expectations.ExpectColumnValuesToBeUnique(column="product_id")),
    ("title_not_null", gx.expectations.ExpectColumnValuesToNotBeNull(column="title_raw")),
    ("title_not_empty", gx.expectations.ExpectColumnValueLengthsToBeBetween(column="title_clean", min_value=1)),
    ("description_not_null", gx.expectations.ExpectColumnValuesToNotBeNull(column="description_raw")),
    ("description_not_empty", gx.expectations.ExpectColumnValueLengthsToBeBetween(column="description_clean", min_value=1)),
])

if reviews is not None:
    reviews["review_score_parsed"] = pd.to_numeric(reviews["review_score"], errors="coerce")
    reviews["review_text_clean"] = reviews["review_text"].fillna("").astype(str).str.strip()
    run_expectations("reviews", reviews, [
        ("review_id_unique", gx.expectations.ExpectColumnValuesToBeUnique(column="review_id")),
        ("review_score_valid_numeric", gx.expectations.ExpectColumnValuesToNotBeNull(column="review_score_parsed")),
        ("review_score_range", gx.expectations.ExpectColumnValuesToBeBetween(column="review_score_parsed", min_value=1, max_value=5)),
        ("review_text_not_null", gx.expectations.ExpectColumnValuesToNotBeNull(column="review_text")),
        ("review_text_not_empty", gx.expectations.ExpectColumnValueLengthsToBeBetween(column="review_text_clean", min_value=1)),
    ])

# 4. Checagem de Integridade Referencial
order_ids = set(orders["order_id"].dropna())
product_ids = set(products["product_id"].dropna())

orphan_item_orders = int((~items["order_id"].isin(order_ids)).sum())
orphan_item_products = int((~items["product_id"].isin(product_ids)).sum())

checks.extend([
    {"table": "items", "expectation": "order_id_referential_integrity", "success": orphan_item_orders == 0, "failed_rows": orphan_item_orders},
    {"table": "items", "expectation": "product_id_referential_integrity", "success": orphan_item_products == 0, "failed_rows": orphan_item_products},
])

if reviews is not None:
    orphan_review_orders = int((~reviews["order_id"].isin(order_ids)).sum())
    checks.append({
        "table": "reviews",
        "expectation": "order_id_referential_integrity",
        "success": orphan_review_orders == 0,
        "failed_rows": orphan_review_orders
    })

# 5. Consolidação dos Resultados e Gravação
ended_at = datetime.now(timezone.utc)
row_counts = {"orders": len(orders), "items": len(items), "products": len(products)}
if reviews is not None:
    row_counts["reviews"] = len(reviews)

failed_checks = [check for check in checks if not check["success"]]
status = "SUCESSO" if not failed_checks else "ERRO"
rows_processed = sum(row_counts.values())

report = {
    "data_inicio": started_at.isoformat(),
    "data_fim": ended_at.isoformat(),
    "status": status,
    "linhas_afetadas": rows_processed,
    "contagens": row_counts,
    "checks": checks
}

report_path = BASE_DIR / "data_quality_report.json"
log_path = BASE_DIR / "log_execucao.csv"

report_path.write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")
pd.DataFrame([{
    "data_inicio": started_at.isoformat(),
    "data_fim": ended_at.isoformat(),
    "status": status,
    "linhas_afetadas": rows_processed,
    "checks_com_falha": len(failed_checks)
}]).to_csv(log_path, index=False)

display(pd.DataFrame(checks))
print(report)

# Baixa os dois artefatos automaticamente no Google Colab.
if IN_COLAB:
    files.download(str(report_path))
    files.download(str(log_path))

Calculating Metrics: 100%|██████████| 11/11 [00:00<00:00, 193.43it/s]


,table,expectation,success,failed_rows
0,orders,order_id_not_null,True,NaN
1,orders,order_id_unique,True,NaN
2,orders,customer_id_not_null,True,NaN
3,orders,purchase_timestamp_not_null,True,NaN
4,orders,purchase_timestamp_valid_datetime,True,NaN
5,orders,order_status_domain,True,NaN
6,items,order_id_not_null,True,NaN
7,items,product_id_not_null,True,NaN
8,items,price_valid_numeric,True,NaN
9,items,price_nonnegative,True,NaN


{'data_inicio': '2026-10-02T13:45:31.040889+00:00', 'data_fim': '2026-10-02T13:45:38.293662+00:00', 'status': 'SUCESSO', 'linhas_afetadas': 315000, 'contagens': {'orders': 100000, 'items': 200000, 'products': 5000, 'reviews': 10000}, 'checks': [{'table': 'orders', 'expectation': 'order_id_not_null', 'success': True}, {'table': 'orders', 'expectation': 'order_id_unique', 'success': True}, {'table': 'orders', 'expectation': 'customer_id_not_null', 'success': True}, {'table': 'orders', 'expectation': 'purchase_timestamp_not_null', 'success': True}, {'table': 'orders', 'expectation': 'purchase_timestamp_valid_datetime', 'success': True}, {'table': 'orders', 'expectation': 'order_status_domain', 'success': True}, {'table': 'items', 'expectation': 'order_id_not_null', 'success': True}, {'table': 'items', 'expectation': 'product_id_not_null', 'success': True}, {'table': 'items', 'expectation': 'price_valid_numeric', 'success': True}, {'table': 'items', 'expectation': 'price_nonnegative', 'suc